# PATE Challenge: EMNIST Letters

This challenge notebook implements the HTB PATE workflow for EMNIST Letters. The validator requires the exact MLP architecture, StandardScaler preprocessing, accuracy of at least 80%, membership-inference advantage of at most 3%, and a Safetensors file named `pate_student.safetensors`.

The notebook keeps submission disabled until local metrics pass. It trains disjoint teachers on private partitions, labels public data through noisy voting, trains the student only on those noisy labels, evaluates accuracy and MIA, saves the artifact, and finally provides an explicit submission cell.

In [ ]:
# Challenge target and controls.
INSTANCE_HOST = '154.57.164.77:34802'
BASE_URL = f'http://{INSTANCE_HOST}'
MODEL_PATH = 'pate_student.safetensors'
SUBMIT = False  # Enable only after local metrics pass and you authorize submission.

RANDOM_SEED = 1337
NUM_CLASSES = 26
NUM_TEACHERS = 10
TEACHER_EPOCHS = 40
STUDENT_EPOCHS = 40
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NOISE_SCALE = 1.0
NUM_STUDENT_QUERIES = 24960
RUN_TRAINING = True
USE_CUDA = False  # CPU is slower but avoids native CUDA/Python crashes on this Windows setup.

## Why this design matches the course

PATE protects the private training set through architectural separation. Each teacher sees one disjoint private partition. The public query set is labeled by noisy teacher votes, and the student trains only on those released labels. The student therefore has no direct training connection to the private records used by the teachers.

The vote mechanism is `y-hat = argmax_j(n_j(x) + Laplace(0, b))`, read aloud as “y-hat equals arg-max over jay of n sub jay of x plus Laplace noise with scale b.” With sensitivity 2 and `b = 20`, the per-query privacy parameter is approximately `epsilon_0 = 2 / 20 = 0.10`.

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from safetensors.torch import save_file, load_file

# Some torchvision builds omit the optional NMS operator at import time.
try:
    _tv_compat = torch.library.Library('torchvision', 'DEF')
    _tv_compat.define('nms(Tensor boxes, Tensor scores, float iou_threshold) -> Tensor')
except Exception:
    pass
from torchvision import datasets

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)
# Bound BLAS/OpenMP parallelism before training so peak memory stays stable.
torch.set_num_threads(1)
DEVICE = torch.device('cuda' if USE_CUDA and torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

## Exact validator architecture

The challenge specifies an MLP with input size 784, hidden layers `[256, 128]`, dropout 0.2, and 26 output logits. EMNIST images have 28 by 28 pixels, so flattening gives `28 * 28 = 784` input features. Labels are converted from EMNIST's 1–26 convention to zero-based 0–25.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_size=784, hidden_layers=None, num_classes=26, dropout=0.2):
        super().__init__()
        hidden_layers = [256, 128] if hidden_layers is None else hidden_layers
        self.layers = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        previous = input_size
        for hidden_size in hidden_layers:
            self.layers.append(nn.Linear(previous, hidden_size))
            self.dropouts.append(nn.Dropout(dropout))
            previous = hidden_size
        self.output = nn.Linear(previous, num_classes)

    def forward(self, x):
        for layer, dropout in zip(self.layers, self.dropouts):
            x = dropout(F.relu(layer(x)))
        return self.output(x)

In [ ]:
# Challenge preprocessing: exact EMNIST split, flattening, scaling, and labels.
train_dataset = datasets.EMNIST('data', split='letters', train=True, download=True)
test_dataset = datasets.EMNIST('data', split='letters', train=False, download=True)
X_train = train_dataset.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
X_test = test_dataset.data.numpy().reshape(-1, 784).astype(np.float32) / 255.0
y_train = train_dataset.targets.numpy().astype(np.int64) - 1
y_test = test_dataset.targets.numpy().astype(np.int64) - 1

scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

# Private data trains teachers; public data is queried for student labels.
X_private, X_public, y_private, y_public = train_test_split(
    X_train_norm, y_train, test_size=0.20, random_state=RANDOM_SEED, stratify=y_train
)
print('Private:', len(X_private), 'Public:', len(X_public), 'Holdout:', len(X_test_norm))

## Disjoint teacher partitions

A single private record must influence only one teacher. We shuffle private indices and slice them into 250 non-overlapping partitions. The final partition receives any remainder.

In [ ]:
partition_indices = np.random.permutation(len(X_private))
partition_size = len(X_private) // NUM_TEACHERS
teacher_partitions = []
for teacher_index in range(NUM_TEACHERS):
    start = teacher_index * partition_size
    stop = len(X_private) if teacher_index == NUM_TEACHERS - 1 else start + partition_size
    teacher_partitions.append(partition_indices[start:stop])
all_partition_indices = np.concatenate(teacher_partitions)
assert len(np.unique(all_partition_indices)) == len(X_private)
print('Partition size:', len(teacher_partitions[0]), 'teachers:', len(teacher_partitions))

In [ ]:
def make_loader(X, y, batch_size=BATCH_SIZE, shuffle=True):
    tensor_x = torch.tensor(X, dtype=torch.float32)
    tensor_y = torch.tensor(y, dtype=torch.long)
    return torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(tensor_x, tensor_y),
        batch_size=batch_size, shuffle=shuffle, num_workers=0,
        pin_memory=(DEVICE.type == 'cuda')
    )

def train_one_model(model, loader, epochs):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
    return model

def predict_classes(model, X):
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(X), 1024):
            x = torch.tensor(X[start:start + 1024], dtype=torch.float32, device=DEVICE)
            outputs.append(model(x).argmax(dim=1).cpu().numpy())
    return np.concatenate(outputs)

## Train teachers and aggregate noisy votes

Each teacher is freshly initialized and trained only on its own partition. We store CPU state dictionaries so 250 models do not occupy GPU memory simultaneously. The public query set is capped by `NUM_STUDENT_QUERIES`; the student never receives the private labels.

In [ ]:
def train_teachers():
    states = []
    for i, indices in enumerate(teacher_partitions):
        # Teachers may be stronger than the required student; each still sees
        # only its own disjoint private partition. This improves vote quality
        # without changing the validator-facing student architecture.
        teacher = MLP(hidden_layers=[512, 256], dropout=0.0).to(DEVICE)
        train_one_model(teacher, make_loader(X_private[indices], y_private[indices]), TEACHER_EPOCHS)
        states.append({k: v.detach().cpu().clone() for k, v in teacher.state_dict().items()})
        del teacher
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
        if (i + 1) % 25 == 0:
            print(f'Teachers trained: {i + 1}/{NUM_TEACHERS}')
    return states

def collect_votes(states, X):
    votes = np.zeros((len(X), NUM_CLASSES), dtype=np.int32)
    # Recreate the same wider teacher architecture used during training.
    # The student architecture remains MLP() with [256, 128] hidden units.
    model = MLP(hidden_layers=[512, 256], dropout=0.0).to(DEVICE)
    for i, state in enumerate(states):
        model.load_state_dict(state)
        predictions = predict_classes(model, X)
        votes[np.arange(len(X)), predictions] += 1
        if (i + 1) % 25 == 0:
            print(f'Votes collected: {i + 1}/{NUM_TEACHERS}')
    return votes

if RUN_TRAINING:
    teacher_states = train_teachers()
    query_count = min(NUM_STUDENT_QUERIES, len(X_public))
    query_indices = np.random.choice(len(X_public), query_count, replace=False)
    X_query, y_query_true = X_public[query_indices], y_public[query_indices]
    vote_counts = collect_votes(teacher_states, X_query)
    noise = np.random.laplace(0.0, NOISE_SCALE, size=vote_counts.shape)
    noisy_labels = np.argmax(vote_counts.astype(np.float64) + noise, axis=1)
    # Diagnostics separate the ensemble's label quality from the student's learning.
    # This is not used as a training label: y_query_true is public-held-out truth
    # used only to explain whether errors come from teachers or from the student.
    clean_labels = vote_counts.argmax(axis=1)
    clean_vote_accuracy = float((clean_labels == y_query_true).mean())
    noisy_vote_accuracy = float((noisy_labels == y_query_true).mean())
    print('Query samples:', query_count, 'vote matrix:', vote_counts.shape)
    print(f'Clean majority-vote accuracy: {clean_vote_accuracy:.4f}')
    print(f'Noisy released-label accuracy: {noisy_vote_accuracy:.4f}')
else:
    print('Training disabled.')

## Train and save the student

The student uses ordinary cross-entropy on noisy public labels. No DP optimizer is needed here: the student never trains on private examples. This is the PATE information bottleneck in code.

In [ ]:
if RUN_TRAINING:
    student_model = train_one_model(MLP(), make_loader(X_query, noisy_labels), STUDENT_EPOCHS)
    student_model.eval()
    test_predictions = predict_classes(student_model, X_test_norm)
    TEST_ACCURACY = float((test_predictions == y_test).mean())
    save_file({k: v.detach().cpu() for k, v in student_model.state_dict().items()}, MODEL_PATH)
    print(f'Test accuracy: {TEST_ACCURACY:.4f}')
    print('Saved:', MODEL_PATH)
else:
    print('Student training disabled.')

## Local membership-inference diagnostic

We compare confidence on private samples versus untouched test samples. The attack searches thresholds and reports `advantage = accuracy - 0.5`. The server validator is authoritative, but this local check catches accidental direct training on private data.

In [ ]:
def mia_advantage(model, members, nonmembers):
    model.eval()
    def confidence(X):
        values = []
        with torch.no_grad():
            for start in range(0, len(X), 1024):
                x = torch.tensor(X[start:start + 1024], dtype=torch.float32, device=DEVICE)
                values.append(torch.softmax(model(x), dim=1).max(dim=1).values.cpu().numpy())
        return np.concatenate(values)
    member_conf, nonmember_conf = confidence(members), confidence(nonmembers)
    n = min(len(member_conf), len(nonmember_conf))
    scores = np.concatenate([member_conf[:n], nonmember_conf[:n]])
    labels = np.concatenate([np.ones(n), np.zeros(n)])
    thresholds = np.percentile(scores, np.linspace(0, 100, 1000))
    accuracy = max(np.mean((scores >= t).astype(int) == labels) for t in thresholds)
    return float(accuracy), float(accuracy - 0.5)

if RUN_TRAINING:
    MIA_ACCURACY, MIA_ADVANTAGE = mia_advantage(student_model, X_private, X_test_norm)
    print(f'MIA accuracy: {MIA_ACCURACY:.4f}; advantage: {MIA_ADVANTAGE:.4f}')
    print('Accuracy requirement met:', TEST_ACCURACY >= 0.80)
    print('MIA requirement met:', MIA_ADVANTAGE <= 0.03)
else:
    print('MIA diagnostic disabled.')

## Submission (disabled by default)

This final cell sends only `pate_student.safetensors` to the authorized target. It checks health first and remains inactive until `SUBMIT = True`.

In [ ]:
if not SUBMIT:
    print('Submission disabled. Set SUBMIT = True after local metrics pass.')
else:
    import requests
    health = requests.get(f'{BASE_URL}/health', timeout=30)
    health.raise_for_status()
    print('Health:', health.json())
    with open(MODEL_PATH, 'rb') as model_file:
        response = requests.post(
            f'{BASE_URL}/validate',
            files={'model': ('pate_student.safetensors', model_file, 'application/octet-stream')},
            timeout=600,
        )
    print('Validator:', response.json())
    response.raise_for_status()